# 2. Factors behind visitation

This notebook tests whether verified NPS metadata adds explanatory information. The available fields are designation, latitude, longitude, and year. Acreage, population proximity, and travel accessibility are not present in the verified sources, so those questions remain unresolved rather than receiving unsupported proxies.

In [1]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.analysis import load_recreation_visits, load_park_metadata, annual_visits, add_metadata, grouped_regression, recent_cohort
raw = ROOT / 'data' / 'raw'
visits = load_recreation_visits(raw / 'Main_Data.csv')
metadata = load_park_metadata(raw / 'nps_parks_api.json')
annual = add_metadata(annual_visits(visits), metadata)
cohort = recent_cohort(annual)
print(f"Complete 2023–2025 cohort: {len(cohort)} units")
display(cohort.groupby('Designation').RecreationVisits.agg(['size', 'median', 'min', 'max']).sort_values('median', ascending=False))
display(cohort.loc[cohort.FullName.isna(), ['UnitCode', 'RecreationVisits']])

Complete 2023–2025 cohort: 394 units


/var/folders/vp/wjhhgw095fj3cnz55r0q960c0000gn/T/ipykernel_66025/3450009015.py:9: UserWarning: Quarantining 4 negative TRV records; affected annual totals are partial
  visits = load_recreation_visits(raw / 'Main_Data.csv')


,size,median,min,max
Designation,,,,
Parkway,2,1.202825e+07,7.381490e+06,1.667501e+07
Memorial Parkway,1,6.968970e+06,6.968970e+06,6.968970e+06
National Seashore,10,2.226613e+06,6.627233e+04,7.885319e+06
National River,1,1.543020e+06,1.543020e+06,1.543020e+06
National Scenic Riverways,1,1.264917e+06,1.264917e+06,1.264917e+06
National Recreation Area,16,1.163888e+06,2.013967e+04,1.596336e+07
Ecological & Historic Preserve,1,1.022139e+06,1.022139e+06,1.022139e+06
National Lakeshore,3,9.326523e+05,2.539597e+05,1.623340e+06
National Scenic Riverway,1,8.531270e+05,8.531270e+05,8.531270e+05


,UnitCode,RecreationVisits
115,FOCA,3.178457e+05
193,JEFM,2.817117e+06
197,JODR,1.446554e+06
209,KICA,7.074150e+05
217,LACH,3.656667e+04
263,NCPC,1.041166e+06
264,NCPE,1.257732e+06
274,OBRI,2.668883e+05
287,PERL,1.605248e+06
298,PRPA,8.418690e+05


## Unit-held-out regression
Use one mean annual total per unit for 2023–2025, aligning the comparison more closely with current metadata. Compare a training-mean baseline, geography, designation, and both on identical five-fold splits repeated five times (seed 42). Ridge regression uses a fixed penalty of 1 to stabilize rare categories; it is not tuned on test data. All preprocessing fits only training units. Zeros are retained with log1p; a positive-only sensitivity shows their influence. Missing coordinates or designation exclude units from this benchmark, not the national totals.

In [2]:
metrics = grouped_regression(annual)
display(metrics.groupby(['sensitivity', 'model'])[['r2_log_visits', 'mae_log_visits']].agg(['mean', 'std', 'min', 'max']))

r2_log_visits                                \
                                   mean       std       min       max   
sensitivity   model                                                     
include_zero  baseline        -0.015505  0.022912 -0.089726 -0.000051   
              combined         0.302839  0.102631  0.091143  0.435827   
              designation      0.273282  0.105905  0.036047  0.414956   
              geography       -0.004669  0.053848 -0.176183  0.062756   
positive_only baseline        -0.015505  0.022912 -0.089726 -0.000051   
              combined         0.302839  0.102631  0.091143  0.435827   
              designation      0.273282  0.105905  0.036047  0.414956   
              geography       -0.004669  0.053848 -0.176183  0.062756   

                          mae_log_visits                                
                                    mean       std       min       max  
sensitivity   model                                                     
include_zero  baseline          1.628313  0.113775  1.410363  1.887660  
              combined          1.282454  0.105861  1.112530  1.454923  
              designation       1.295140  0.114203  1.095802  1.501631  
              geography         1.610758  0.113334  1.404700  1.849279  
positive_only baseline          1.628313  0.113775  1.410363  1.887660  
              combined          1.282454  0.105861  1.112530  1.454923  
              designation       1.295140  0.114203  1.095802  1.501631  
              geography         1.610758  0.113334  1.404700  1.849279

## Interpretation and limitations
R² and MAE are on log1p mean annual visits, not visit counts. Negative R² means worse than the held-out sample mean. The dummy baseline predicts the training mean. Fold variation is split sensitivity, not a confidence interval: repeated folds overlap. See reports/model_summary.csv for exact results. Designation medians are the main descriptive result; prediction is a secondary check, not causal explanation. Coordinates are representative points, not access or population measures. Current designations are not historical classifications.

In [3]:
display(cohort[['RecreationVisits', 'Latitude', 'Longitude']].corr(numeric_only=True))
print('Acreage, nearby population, accessibility and counting-method changes remain unresolved.')

,RecreationVisits,Latitude,Longitude
RecreationVisits,1.000000,-0.032453,0.038786
Latitude,-0.032453,1.000000,-0.264014
Longitude,0.038786,-0.264014,1.000000


Acreage, nearby population, accessibility and counting-method changes remain unresolved.
